# Removing Warm ACIS Data


In [1]:
#source /export/ciao/bin/ciao.bash -o
PFILES="./param;$ASCDS_INSTALL/param:$ASCDS_INSTALL/contrib/param"
ASCDS_WORK_PATH=./
mkdir -p ./param

In [2]:
/bin/rm -rf 17233
download_chandra_obsid 17233 evt2,mtl,asol,bpix,msk
mv 17233/*/*fits.gz .
gunzip -f *fits.gz
/bin/rm -rf 17233


  Type     Format      Size  0........H.........1  Download Time Average Rate
  ---------------------------------------------------------------------------
  asol     fits        9 Mb  ####################          < 1 s  94743.5 kb/s
  evt2     fits        5 Mb  ####################          < 1 s  77842.7 kb/s
  mtl      fits        2 Mb  ####################          < 1 s  69402.6 kb/s
  bpix     fits       56 Kb  ####################          < 1 s  4396.2 kb/s
  msk      fits        5 Kb  ####################          < 1 s  511.6 kb/s

      Total download size for ObsId 17233 = 16 Mb
      Total download time for ObsId 17233 = < 1 s



In [3]:
dmstat 'acisf17233_000N002_mtl1.fits[cols FP_TEMP]'

FP_TEMP
    min:	153.60722351 	      @:	333 
    max:	157.96488953 	      @:	13607 
   mean:	155.79074933 
  sigma:	1.3777218286 
    sum:	2165335.6249 
   good:	13899 
   null:	0 



In [4]:
dmstat infile='acisf17233N002_evt2.fits[sky=circle(4184,3915,10)][cols ccd_id]'

ccd_id
    min:	3 	      @:	1 
    max:	3 	      @:	1 
   mean:	3 
  sigma:	0 
    sum:	13836 
   good:	4612 
   null:	0 



In [5]:
python << EOM
from pycrates import read_file
import matplotlib.pyplot as plt

mtl=read_file("acisf17233_000N002_mtl1.fits[cols time,fp_temp]")
times = mtl.get_column("time").values
time_offset = (times - times[0]) / 1000.0  # sec -> ksec
fp_temp = mtl.get_column("fp_temp").values

p = plt.plot(time_offset,fp_temp)
p[0].set_marker("o")
p[0].set_markersize(1)
p[0].set_linestyle('None')

plt.xlabel("Time since observation start [ks]")
plt.ylabel("FP_TEMP [Kelvin]")
plt.title("OBS_ID : 17233")

plt.savefig("obs17233_fptemp.png")
EOM



Qt: Session management error: None of the authentication protocols specified are supported


In [6]:
dmgti infile='acisf17233_000N002_mtl1.fits' outfile=temp_153K_154K_GTI.fits \
      userlimit='( (FP_TEMP >= 152.96 ) && (FP_TEMP < 153.96 ) )' clobber=yes
dmgti infile='acisf17233_000N002_mtl1.fits' outfile=temp_154K_156K_GTI.fits \
      userlimit='( (FP_TEMP >= 153.96 ) && (FP_TEMP < 155.96 ) )' clobber=yes
dmgti infile='acisf17233_000N002_mtl1.fits' outfile=temp_156K_158K_GTI.fits \
      userlimit='( (FP_TEMP >= 155.96 ) && (FP_TEMP < 157.965 ) )' clobber=yes


Input MTL file: acisf17233_000N002_mtl1.fits
Output gti file name: temp_153K_154K_GTI.fits
Output mtl file name: none
Output GTI file: temp_153K_154K_GTI.fits
Input MTL file: acisf17233_000N002_mtl1.fits
Output gti file name: temp_154K_156K_GTI.fits
Output mtl file name: none
Output GTI file: temp_154K_156K_GTI.fits
Input MTL file: acisf17233_000N002_mtl1.fits
Output gti file name: temp_156K_158K_GTI.fits
Output mtl file name: none
Output GTI file: temp_156K_158K_GTI.fits


In [7]:
dmcopy infile='acisf17233N002_evt2.fits[@temp_153K_154K_GTI.fits]' outfile='obs17233_temp_153K_154K_evt2.fits' clob+
dmhedit infile='obs17233_temp_153K_154K_evt2.fits' filelist='None' operation='add' key='FP_TEMP' value=153.5

dmcopy infile='acisf17233N002_evt2.fits[@temp_154K_156K_GTI.fits]' outfile='obs17233_temp_154K_156K_evt2.fits' clob+
dmhedit infile='obs17233_temp_154K_156K_evt2.fits' filelist='None' operation='add' key='FP_TEMP' value=155.0

dmcopy infile='acisf17233N002_evt2.fits[@temp_156K_158K_GTI.fits]' outfile='obs17233_temp_156K_158K_evt2.fits' clob+
dmhedit infile='obs17233_temp_156K_158K_evt2.fits' filelist='None' operation='add' key='FP_TEMP' value=157.0


In [8]:
dmlist acisf17233N002_evt2.fits header | grep EXPOSURE

0113 EXPOSURE                 42520.878762310 [s]        Real8        Exposure time


In [9]:
dmlist obs17233_temp_153K_154K_evt2.fits header | grep EXPOSURE
dmlist obs17233_temp_154K_156K_evt2.fits header | grep EXPOSURE
dmlist obs17233_temp_156K_158K_evt2.fits header | grep EXPOSURE

0113 EXPOSURE                  6262.7888060302 [s]       Real8        Exposure time
0113 EXPOSURE                 15438.218033820 [s]        Real8        Exposure time
0113 EXPOSURE                 20819.871922460 [s]        Real8        Exposure time


In [10]:
specextract infile='obs17233_temp_153K_154K_evt2.fits[sky=circle(4184,3915,10.0)]' \
      bkgfile='obs17233_temp_153K_154K_evt2.fits[sky=circle(4278,3642,40.0)]' \
      outroot=obs17233_temp_153K_154K weight_rmf=yes weight=yes clob+

specextract infile='obs17233_temp_154K_156K_evt2.fits[sky=circle(4184,3915,10.0)]' \
      bkgfile='obs17233_temp_154K_156K_evt2.fits[sky=circle(4278,3642,40.0)]' \
      outroot=obs17233_temp_154K_156K weight_rmf=yes weight=yes clob+

specextract infile='obs17233_temp_156K_158K_evt2.fits[sky=circle(4184,3915,10.0)]' \
      bkgfile='obs17233_temp_156K_158K_evt2.fits[sky=circle(4278,3642,40.0)]' \
      outroot=obs17233_temp_156K_158K weight_rmf=yes weight=yes clob+

Running specextract
Version: 4 September 2025

Extracting Spectra:[███████████████████████████████████████████████████████] 2/2

Generating Aspect Histograms & Weights Maps:[██████████████████████████████] 2/2

Generating ARFs & RMFs:[███████████████████████████████████████████████████] 4/4

Grouping Spectra:[█████████████████████████████████████████████████████████] 2/2

Adding Header Keywords:[███████████████████████████████████████████████████] 2/2

Running specextract
Version: 4 September 2025

Extracting Spectra:[███████████████████████████████████████████████████████] 2/2

Generating Aspect Histograms & Weights Maps:[██████████████████████████████] 2/2

Generating ARFs & RMFs:[███████████████████████████████████████████████████] 4/4

Grouping Spectra:[█████████████████████████████████████████████████████████] 2/2

Adding Header Keywords:[███████████████████████████████████████████████████] 2/2

Running specextract
Version: 4 September 2025

Extracting Spectra:[███████████████████

In [11]:
dmlist obs17233_temp_156K_158K.rmf header | grep SCATFILE


0034 SCATFILE             /data/chandra_caldb/ciao/data/chandra/acis/p2_resp/acisD2000-01-29p2_respN0009_115-117.fits String       Scatter matrix file


In [12]:
/bin/ls obs17233_temp*_evt2.fits > evt_stack.lis
cat evt_stack.lis

obs17233_temp_153K_154K_evt2.fits
obs17233_temp_154K_156K_evt2.fits
obs17233_temp_156K_158K_evt2.fits


In [13]:
specextract infile="@evt_stack.lis[sky=circle(4184,3915,10.0)]" \
  bkgfile="@evt_stack.lis[sky=circle(4278,3642,40.0)]" \
  outroot=obs17233_temps weight=yes weight_rmf=yes combine=yes clob+


Running specextract
Version: 4 September 2025

Extracting Spectra:[███████████████████████████████████████████████████████] 6/6

Generating Aspect Histograms & Weights Maps:[██████████████████████████████] 6/6

Generating ARFs & RMFs:[█████████████████████████████████████████████████] 12/12

Grouping Spectra:[█████████████████████████████████████████████████████████] 6/6

Adding Header Keywords:[███████████████████████████████████████████████████] 6/6



In [14]:
acis_split_evt_by_fptemp acisf17233N002_evt2.fits out=auto_split clob+

acis_split_evt_by_fptemp (13 April 2025)
          infile = acisf17233N002_evt2.fits
         outroot = auto_split
         mtlfile = INDEF
         clobber = yes
         verbose = 1
            mode = ql

FP_TEMP [K]: 165.96 - 178.16 :       0.00 [sec]
FP_TEMP [K]: 163.96 - 165.96 :       0.00 [sec]
FP_TEMP [K]: 161.96 - 163.96 :       0.00 [sec]
FP_TEMP [K]: 159.96 - 161.96 :       0.00 [sec]
FP_TEMP [K]: 157.96 - 159.96 :      46.50 [sec]
FP_TEMP [K]: 155.96 - 157.96 :   21049.00 [sec]
FP_TEMP [K]: 153.96 - 155.96 :   15642.60 [sec]
FP_TEMP [K]: 152.96 - 153.96 :    6348.80 [sec]

Created the following event files:
    auto_split_157.96-159.96.evt
    auto_split_155.96-157.96.evt
    auto_split_153.96-155.96.evt
    auto_split_152.96-153.96.evt

Use '@auto_split_evt.lis' to extract spectra and responses.


In [15]:
cat auto_split_evt.lis

!auto_split_157.96-159.96.evt
!auto_split_155.96-157.96.evt
!auto_split_153.96-155.96.evt
!auto_split_152.96-153.96.evt


In [16]:
dmlist @auto_split_evt.lis header | grep EXPOSURE

0113 EXPOSURE                    45.8924436202 [s]       Real8        Exposure time
0113 EXPOSURE                 20773.979478840 [s]        Real8        Exposure time
0113 EXPOSURE                 15438.218033820 [s]        Real8        Exposure time
0113 EXPOSURE                  6262.7888060302 [s]       Real8        Exposure time


In [20]:
dmlist @auto_split_evt.lis"[sky=circle(4184,3915,10.0)]" counts

5       
2232    
1698    
677     


In [21]:
dmlist @auto_split_evt.lis"[sky=circle(4278,3642,40.0)]" counts

0       
81      
61      
33      


In [17]:
specextract @auto_split_evt.lis"[sky=circle(4184,3915,10.0)]" \
  bkgfile="@auto_split_evt.lis[sky=circle(4278,3642,40.0)]" \
  outroot=auto_split_temps weight=yes weight_rmf=yes combine=yes clob+ || echo

Running specextract
Version: 4 September 2025

# specextract (4 September 2025): ERROR auto_split_157.96-159.96.evt[sky=circle(4278,3642,40.0)] has zero counts in the 'energy_wmap=300:2000' eV range needed to generate a weights map.



In [18]:
tail -3 auto_split_evt.lis > tweak_auto.lis
cat tweak_auto.lis

!auto_split_155.96-157.96.evt
!auto_split_153.96-155.96.evt
!auto_split_152.96-153.96.evt


In [19]:
specextract "@tweak_auto.lis[sky=circle(4184,3915,10.0)]" \
  bkgfile="@tweak_auto.lis[sky=circle(4278,3642,40.0)]" \
  outroot=auto_split_temps weight=yes weight_rmf=yes combine=yes clob+

Running specextract
Version: 4 September 2025

Extracting Spectra:[███████████████████████████████████████████████████████] 6/6

Generating Aspect Histograms & Weights Maps:[██████████████████████████████] 6/6

Generating ARFs & RMFs:[█████████████████████████████████████████████████] 12/12

Grouping Spectra:[█████████████████████████████████████████████████████████] 6/6

Adding Header Keywords:[███████████████████████████████████████████████████] 6/6

